In [0]:
%sql
-- CREATE EXTERNAL VOLUME dbcertification.easewithdata.volume_easewithdata
-- LOCATION 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/volume_ewd'
-- comment 'This is landing Managed Volume' ;

In [0]:
dbutils.fs.mkdirs("/Volumes/dbcertification/easewithdata/volume_easewithdata/input")

In [0]:
dbutils.fs.cp("/Volumes/samples/databricks/datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv","/Volumes/dbcertification/easewithdata/volume_easewithdata/input")

In [0]:
dbutils.fs.cp("/Volumes/samples/databricks/datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv","/Volumes/dbcertification/easewithdata/volume_easewithdata/input")

#### What is a placeholder/empty table? 
- Tables that can be used with **COPY INTO, INSERT INTO and MERGE INTO** Commands
- For these files schema is created on the fly

In [0]:
%sql
-- CREATE PLACEHOLDER TABLE
create table dbcertification.easewithdata.invoice_cp
using delta
location 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoices_cp'

In [0]:
%sql
-- use COPY INTO to load data in placeholder table
 COPY INTO dbcertification.easewithdata.invoice_cp
 from "/Volumes/dbcertification/easewithdata/volume_easewithdata/input"
 fileformat = CSV
 pattern = '*.csv'
 FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'header' = 'true' 
    )
COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
%sql
SELECT * from dbcertification.easewithdata.invoice_cp;

- **COPY INTO** : This command is idempotent, meaning data/file processed already cannot be processed again.
- Running it again below : 0 records inserted
- COPY INTO is retriable and idempotent command.
- **COPY INTO maintains its metadata at table level,  So if a new table is created data will get loaded again.**
- COPY INTO maintains its metadata in delta_log's copy_into_catalog folder to make sure your pipelines are idempotent.
- Even if you re-run the same file again and again, nothing will be loaded. 

In [0]:
%sql
-- use COPY INTO to load data in placeholder table
 COPY INTO dbcertification.easewithdata.invoice_cp
 from "/Volumes/dbcertification/easewithdata/volume_easewithdata/input"
 fileformat = CSV
 pattern = '*.csv'
 FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'header' = 'true' 
    )
COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
%sql
describe extended dbcertification.easewithdata.invoice_cp;

In [0]:
%sql
-- CREATE new table with only 3 columns and one custome column _insert_date
create table dbcertification.easewithdata.invoice_cp_alt
(
    InvoiceNo string,
    StockCode string,
    Quantity double,
    _insert_date timestamp
)
using delta
location 'abfss://data@dbcertificationstg.dfs.core.windows.net/easewithdata/invoices_cp_alt'

In [0]:
%sql
-- load data using copy into in the new table dbcertification.easewithdata.invoice_cp_alt
COPY INTO dbcertification.easewithdata.invoice_cp_alt
 from (
        select InvoiceNo, StockCode, cast(Quantity as double) as Quantity, current_timestamp as _insert_date
        from 
        "/Volumes/dbcertification/easewithdata/volume_easewithdata/input"
    )
 fileformat = CSV
 pattern = '*.csv'
 FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'header' = 'true' 
    )
;

In [0]:
%sql
select * from dbcertification.easewithdata.invoice_cp_alt; 

In [0]:
dbutils.fs.cp("/Volumes/samples/databricks/datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv","/Volumes/dbcertification/easewithdata/volume_easewithdata/input")

In [0]:
%sql
COPY INTO dbcertification.easewithdata.invoice_cp_alt
 from (
        select InvoiceNo, StockCode, cast(Quantity as double) as Quantity, current_timestamp as _insert_date
        from 
        "/Volumes/dbcertification/easewithdata/volume_easewithdata/input"
    )
 fileformat = CSV
 pattern = '*.csv'
 FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'header' = 'true' 
    )
;

In [0]:
%sql
select count(1) from dbcertification.easewithdata.invoice_cp_alt;

- Data is processed Incrementally
- Only new records are added.
- Very Useful when you have to load 1000 files.
- But if you have millions of files to load per hour or have very complex nested directory structure , those cases databricks recommends to use AutoLoaders.